# RETO NLP!

Tu objetivo: conseguir el mejor **F1-score** posible en el split de **test**.

## Contexto

Tienes dos tipos de datos:

| Datos | Formato | Etiquetas |
|---|---|---|
| **WikiANN (es)** (train/dev/test) | palabras + etiquetas BIO | ✅ Sí |
| **Transcripciones de audio** (`.json`) | texto libre | ❌ No |

Los datos no etiquetados son transcripciones reales de entrevistas en español (JEP). No tienen anotaciones, pero contienen entidades, depende de ti decidir cómo aprovecharlos.

Una estrategia posible es generar **pseudo-labels** con el modelo entrenado sobre WikiANN y usarlos para seguir entrenando, pero hay total libertad creativa.

La única zona que debes modificar está marcada con ✏️

In [ ]:
!pip install transformers seqeval datasets -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 1. Setup

In [ ]:
import json, re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from seqeval.metrics import f1_score
from datasets import load_dataset

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'dccuchile/bert-base-spanish-wwm-cased'
MAX_LEN    = 128
BATCH_SIZE = 16

LABEL2ID = {'O':0,'B-PER':1,'I-PER':2,'B-ORG':3,'I-ORG':4,'B-LOC':5,'I-LOC':6}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Device:', DEVICE)

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Device: cuda


## 2. Datos etiquetados - WikiANN (español)

Dataset público multilingüe de NER basado en Wikipedia.
Incluye etiquetas BIO para `PER`, `ORG` y `LOC`.
El split de **test** es el que se usa para la evaluación final, **no lo uses en el entrenamiento**.

In [ ]:
raw_wiki = load_dataset('unimelb-nlp/wikiann', 'es')

# WikiANN labels: ['O','B-PER','I-PER','B-ORG','I-ORG','B-LOC','I-LOC']
WIKI_LABELS = raw_wiki['train'].features['ner_tags'].feature.names

def wikiann_to_list(split):
    """Convierte un split de WikiANN al formato [(palabras, etiquetas), ...]"""
    data = []
    for ex in split:
        words  = ex['tokens']
        labels = [WIKI_LABELS[t] for t in ex['ner_tags']]
        data.append((words, labels))
    return data

train_labeled = wikiann_to_list(raw_wiki['train'])
dev_data      = wikiann_to_list(raw_wiki['validation'])
test_data     = wikiann_to_list(raw_wiki['test'])

print(f'Train: {len(train_labeled)} | Dev: {len(dev_data)} | Test: {len(test_data)}')
print('Ejemplo:', train_labeled[0])

README.md:   0%|          | 0.00/158k [00:00<?, ?B/s]

es/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  608kB            

es/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

es/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  608kB            

es/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

es/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Train: 20000 | Dev: 10000 | Test: 10000
Ejemplo: (['REDIRECCIÓN', 'Algarrobo', '(', 'Chile', ')'], ['O', 'B-LOC', 'I-LOC', 'I-LOC', 'I-LOC'])


## 3. Datos sin etiquetar - Transcripciones de entrevistas

Corpus de entrevistas reales en español (JEP — Jurisdicción Especial para la Paz).
**No tienen etiquetas NER**; cada sección es una entrevista distinta.

El archivo `entrevistas_combinado.json` se encuentra en la carpeta assets de la sesión 11.

In [ ]:
with open('entrevistas_combinado.json', encoding='utf-8') as f:
    corpus_entrevistas = json.load(f)

print(f'{corpus_entrevistas["n_sections"]} secciones / entrevistas cargadas')
print('Descripción:', corpus_entrevistas['description'])
print('\nEjemplo de sección:')
sec0 = corpus_entrevistas['sections'][0]
print(f'  Título : {sec0["title"]}')
print(f'  Chars  : {len(sec0["text"])}')

oraciones = []
for seccion in corpus_entrevistas['sections']:
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', seccion['text']) if len(s.strip()) > 10]
    oraciones.extend(sents)

print(f'\n{len(oraciones)} oraciones extraídas de {corpus_entrevistas["n_sections"]} entrevistas')
print('Ejemplo:', oraciones[0])

72 secciones / entrevistas cargadas
Descripción: Transcripciones de entrevistas JEP — corpus no etiquetado en español

Ejemplo de sección:
  Título : 20220927 ｜ Versión Voluntaria ｜ Fuerza Pública Eduardo León Figueroa Cifuentes
  Chars  : 193744

21902 oraciones extraídas de 72 entrevistas
Ejemplo: Buenos días.


## 4. Dataset y DataLoader

In [ ]:
class NERDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        words, labels = self.data[idx]
        enc = tokenizer(words, is_split_into_words=True, max_length=MAX_LEN,
                        padding='max_length', truncation=True, return_tensors='pt')
        ids, prev, aligned = enc.word_ids(), None, []
        for w in ids:
            aligned.append(LABEL2ID.get(labels[w], 0) if w is not None and w != prev else -100)
            prev = w
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'labels': torch.tensor(aligned)}

train_loader = DataLoader(NERDataset(train_labeled), batch_size=BATCH_SIZE, shuffle=True)
dev_loader   = DataLoader(NERDataset(dev_data),      batch_size=BATCH_SIZE)
test_loader  = DataLoader(NERDataset(test_data),     batch_size=BATCH_SIZE)

## ✏️ 5. Tu módulo

Esta es la zona que puedes modificar.

El módulo se inserta entre BETO y el clasificador final:
```
BETO → [ MiModulo ] → clasificador
       (B, L, 768)    (B, L, 768)
```
El módulo recibe tensores `(batch, seq_len, 768)` y debe devolver el mismo shape.
Puedes añadir las capas que quieras, o dejarlo quieto si prefieres no modificar nada.

In [ ]:
class MiModulo(nn.Module):
    def __init__(self):
        super().__init__()
        # ✏️ define tus capas aquí
        self.red = nn.Sequential(
            nn.Linear(768, 768),
            nn.GELU(),
            nn.Linear(768, 768),
        )

    def forward(self, x):       # x: (batch, seq_len, 768)
        return x + self.red(x)  # ✏️ cambia según desees

mi_modulo = MiModulo().to(DEVICE)
print(mi_modulo)

MiModulo(
  (red): Sequential(
    (0): Linear(in_features=768, out_features=768, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=768, out_features=768, bias=True)
  )
)


## 6. Modelo, training e inferencia

In [ ]:
class ModeloNER(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(MODEL_NAME)
        self.modulo     = mi_modulo
        self.classifier = nn.Linear(768, len(LABEL2ID))

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids, attention_mask).last_hidden_state  # (B, L, 768)
        x = self.modulo(x)                                          # tú módulo modificado
        return self.classifier(x)                                   # (B, L, n_labels)

model     = ModeloNER().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn   = nn.CrossEntropyLoss(ignore_index=-100)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#@title entrenamiento
EPOCHS = 5  #puedes cambiar esto

for epoch in range(EPOCHS):
    # training
    model.train()
    total_loss = 0
    for batch in train_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['labels'].to(DEVICE)
        loss = loss_fn(model(ids, mask).view(-1, len(LABEL2ID)), lbls.view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()

    # eval en dev/validation
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in dev_loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['labels']
            preds = model(ids, mask).argmax(-1).cpu()
            for p, l in zip(preds, lbls):
                preds_all.append([ID2LABEL[i.item()] for i, j in zip(p, l) if j.item() != -100])
                labels_all.append([ID2LABEL[j.item()] for j in l if j.item() != -100])

    f1 = f1_score(labels_all, preds_all)
    print(f'Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.3f} | F1 val: {f1:.3f}')

torch.save(model.state_dict(), 'modelo.pt')
print('modelo guardado en modelo.pt')

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 1 | Loss: 0.283 | F1 val: 0.870
Epoch 2 | Loss: 0.145 | F1 val: 0.884
Epoch 3 | Loss: 0.095 | F1 val: 0.883
Epoch 4 | Loss: 0.066 | F1 val: 0.892
Epoch 5 | Loss: 0.048 | F1 val: 0.891
modelo guardado en modelo.pt


In [ ]:
#@title inferencia
def predecir(texto):
    model.eval()
    words = texto.split()
    enc   = tokenizer(words, is_split_into_words=True, return_tensors='pt',
                      truncation=True, max_length=MAX_LEN).to(DEVICE)
    with torch.no_grad():
        preds = model(enc['input_ids'], enc['attention_mask']).argmax(-1).squeeze().cpu()
    ids, prev, resultado = enc.word_ids(), None, []
    for i, w in enumerate(ids):
        if w is not None and w != prev:
            resultado.append((words[w], ID2LABEL[preds[i].item()]))
        prev = w
    return resultado

for palabra, etiqueta in predecir("Me llamo Andrea y estudio en la Universidad Industrial de Santander"):
    print(f'{palabra:<20} {etiqueta}')

Me                   O
llamo                O
Andrea               B-PER
y                    O
estudio              O
en                   O
la                   O
Universidad          B-ORG
Industrial           I-ORG
de                   I-ORG
Santander            I-ORG


## 7. Evaluación final sobre test

Corre esta celda **cuando tengas tu modelo listo** para entregar. Deberás subir un pantallazo al discord, pero no borres tu notebook!! revisaremos el código de los primeros lugares para decidir.

In [ ]:
from seqeval.metrics import classification_report

checkpoint = torch.load('modelo.pt', map_location=DEVICE)
model.load_state_dict(checkpoint)

model.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for batch in test_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['labels']
        preds = model(ids, mask).argmax(-1).cpu()
        for p, l in zip(preds, lbls):
            preds_all.append([ID2LABEL[i.item()] for i, j in zip(p, l) if j.item() != -100])
            labels_all.append([ID2LABEL[j.item()] for j in l if j.item() != -100])

print(f'F1 en test: {f1_score(labels_all, preds_all):.4f}')
print(classification_report(labels_all, preds_all))

F1 en test: 0.8937
              precision    recall  f1-score   support

         LOC       0.91      0.93      0.92      4725
         ORG       0.85      0.83      0.84      3576
         PER       0.88      0.94      0.91      3959

   micro avg       0.88      0.90      0.89     12260
   macro avg       0.88      0.90      0.89     12260
weighted avg       0.88      0.90      0.89     12260

